# Prompt Chaining and Iterative Generation for Story Writing

This notebook demonstrates how to write a story using two  powerful tools: prompt chaining and iterative generation. These can be used to tackle complex tasks that are difficult or impossible to complete in a single step.

**Prompt chaining** involves breaking down a larger task into smaller, interconnected prompts. The output of each prompt then becomes the input for the next, guiding the language model through the process step-by-step. This approach offers several benefits:

*   Improved accuracy: Smaller, focused prompts can lead to better results from the language model.
*   Debugging: It's easier to identify where things go wrong within the chain, allowing for targeted adjustments and improvements.
*   Complex tasks: By breaking down intricate problems into manageable steps, prompt chaining enables the language model to tackle more complex tasks.

**Iterative generation** refers to the process of building the desired output iteratively. In this case, you will use it to write a story that is longer than what a single generation window allows. Iterative generation offers several benefits:

*   Longer outputs: It allows for the creation of longer and more detailed outputs, exceeding the limitations of a single generation window.
*   Flexibility: You can adjust and refine the output at each iteration, ensuring the story develops in the desired direction.
*   Human-in-the-loop control: You can provide feedback and guidance at each step, ensuring the story aligns with your creative vision.

By combining these techniques, you can create a compelling and well-structured story, piece by piece, while maintaining control over the creative process.


## Setup

In [ ]:
!pip install -q -U openai

from getpass import getpass
from openai import OpenAI
from pprint import pprint

api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

MODEL = "gpt-5-mini"

Enter the OpenAI API key provided for this course when prompted.

The key is entered with `getpass()`, so it will not be displayed in the notebook output.

**OpenAI setup**

This notebook uses the OpenAI Python SDK and the Responses API. All model calls use the same `client` and `MODEL` defined above.

We will use `gpt-5-mini` for this exercise. You only need to change the `MODEL` value in the setup cell if your instructor asks you to use a different OpenAI model.

## Prompts: Guiding the language model

You will use a series of interconnected prompts to guide the language model through the process of writing a story. These prompts will cover the story's premise, outline, and starting point, ultimately leading to the creation of a complete narrative.

It's important to carefully craft these prompts to provide clear instructions and relevant information to the language model. This will help the model generate high-quality content that aligns with your creative vision.


### Prompt chain for story writing

This section contains the prompts that will guide the language model through the story writing process. These prompts are designed to be chained together, with the output of one prompt feeding into the next.

Each prompt includes a **persona statement**, which helps the language model understand its role and generate more relevant and accurate content. In this case, the persona statement is: `You are an award-winning science fiction author with a penchant for expansive, intricately woven stories. Your ultimate goal is to write the next award winning sci-fi novel.`

Since the persona statement and writing guidelines appear in multiple prompts, f-string variables are used to add them to the prompts.

Additionally, the prompts use **placeholders** to insert the results of previous prompts using `.format()`. This allows us to build the story step-by-step, incorporating the outputs generated by the model at each stage. These are normally denoted by `{}`, but since you are also using f-string variables they are escaped as `{{}}`.

Here's a breakdown of the prompt chain:

1.  **Premise prompt**: This prompt asks the model to generate a single-sentence premise for a sci-fi story featuring cats.
1.  **Outline prompt**: This prompt provides the generated premise to the model and asks it to create a plot outline for the story.
1.  **Starting prompt**: This prompt provides both the premise and the outline to the model and asks it to begin writing the story. It also includes instructions to write a detailed and lengthy opening section.

By chaining these prompts together, it guides the language model through the process of creating a well-structured and engaging story.

#### Example

Let's say the model generates the following premise:

> In a future where humanity has achieved interstellar travel, a group of genetically enhanced cats embarks on a perilous mission to save the galaxy from a sinister alien threat.

This premise would then be inserted into the outline prompt:

> You are an award-winning science fiction author with a penchant for expansive,
intricately woven stories. Your ultimate goal is to write the next award winning
sci-fi novel.
>
> You have a gripping premise in mind:
>
> **In a future where humanity has achieved interstellar travel, a group of genetically enhanced cats embarks on a perilous mission to save the galaxy from a sinister alien threat.**
>
> Write an outline for the plot of your story.


The model would then generate an outline based on this premise, which would then be used in the starting prompt to begin writing the story itself.

This process continues iteratively, with the model generating additional content based on the previous outputs, until the story is complete.


In [ ]:
persona = '''\
You are an award-winning science fiction author with a penchant for expansive,
intricately woven stories. Your ultimate goal is to write the next award winning
sci-fi novel.'''

guidelines = '''\
Writing Guidelines

Delve deeper. Lose yourself in the world you're building. Unleash vivid
descriptions to paint the scenes in your reader's mind. Develop your
characters—let their motivations, fears, and complexities unfold naturally.
Weave in the threads of your outline, but don't feel constrained by it. Allow
your story to surprise you as you write. Use rich imagery, sensory details, and
evocative language to bring the setting, characters, and events to life.
Introduce elements subtly that can blossom into complex subplots, relationships,
or worldbuilding details later in the story. Keep things intriguing but not
fully resolved. Avoid boxing the story into a corner too early. Plant the seeds
of subplots or potential character arc shifts that can be expanded later.

Remember, your main goal is to write as much as you can. If you get through
the story too fast, that is bad. Expand, never summarize.
'''

premise_prompt = f'''\
{persona}

Write a single sentence premise for a sci-fi story featuring cats.'''

outline_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Write an outline for the plot of your story.'''

starting_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Your imagination has crafted a rich narrative outline:

{{outline}}

First, silently review the outline and the premise. Consider how to start the
story.

Start to write the very beginning of the story. You are not expected to finish
the whole story now. Your writing should be detailed enough that you are only
scratching the surface of the first bullet of your outline. Try to write AT
MINIMUM 1000 WORDS and MAXIMUM 2000 WORDS.

{guidelines}'''

### Continuation prompt: Building the story

Once the language model has generated the beginning of the story, you can use a **continuation prompt** to iteratively expand the narrative. This prompt is similar to the starting prompt, but with two key differences:

1.  **Instruction to signal completion**: An instruction was added for the model to write `IAMDONE` when it believes the story is finished. This serves as a signal for us to stop generating additional content.
1.  **Work in progress**: The language in the prompt is adjusted to reflect that the story is already in progress, rather than starting from scratch.

The continuation prompt provides the model with the story's premise, outline, and the existing draft. It then instructs the model to continue writing the story in detail.

This iterative process allows us to build a longer and more complex story than what would be possible in a single generation call. You can continue feeding the existing draft back into the continuation prompt until the model signals that the story is complete by writing `IAMDONE`.

**Note**: The `IAMDONE` signal is simply a convenient way to identify the story's end in this example. In other applications of iterative generation, different methods might be used to determine when the desired output is complete.


In [ ]:
continuation_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Your imagination has crafted a rich narrative outline:

{{outline}}

You've begun to immerse yourself in this world, and the words are flowing.
Here's what you've written so far:

{{story_text}}

=====

First, silently review the outline and story so far. Identify what the single
next part of your outline you should write.

Your task is to continue where you left off and write the next part of the story.
You are not expected to finish the whole story now. Your writing should be
detailed enough that you are only scratching the surface of the next part of
your outline. Try to write AT MINIMUM 1000 WORDS. However, only once the story
is COMPLETELY finished, write IAMDONE. Remember, do NOT write a whole chapter
right now.

{guidelines}'''

## Writing time!

### Generate and print the premise

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=premise_prompt,
)

premise = response.output_text
print(premise)

### Generate and print the outline

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=outline_prompt.format(premise=premise),
)

outline = response.output_text
print(outline)

### Generate the start of the story

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=starting_prompt.format(
        premise=premise,
        outline=outline,
    ),
)

starting_draft = response.output_text
pprint(starting_draft)

### Generate the continuation of the story and check progress

In [ ]:
draft = starting_draft

response = client.responses.create(
    model=MODEL,
    input=continuation_prompt.format(
        premise=premise,
        outline=outline,
        story_text=draft,
    ),
)

continuation = response.output_text
pprint(continuation)

### Let's finish writing the story.

The first continuation above shows one iteration manually. Next, we repeat the same process a few more times.

The loop stops when the model writes `IAMDONE`, or when it reaches `MAX_CONTINUATIONS`. The maximum keeps the example predictable and prevents accidental unlimited API calls.

In [ ]:
# Add the first continuation to the draft
draft = draft + "\n\n" + continuation

# Continue the story, but cap the number of additional API calls
MAX_CONTINUATIONS = 5

for _ in range(MAX_CONTINUATIONS):
    if "IAMDONE" in continuation:
        break

    response = client.responses.create(
        model=MODEL,
        input=continuation_prompt.format(
            premise=premise,
            outline=outline,
            story_text=draft,
        ),
    )

    continuation = response.output_text
    draft = draft + "\n\n" + continuation

# Remove the completion signal and print the final story
final = draft.replace("IAMDONE", "").strip()
pprint(final)

Language models have limits on how much text they can generate in a single response. Iterative generation avoids asking the model to write the entire story at once. Instead, each call adds another section to the existing draft.

This is the key idea: **generate → append → continue** until the task is complete or you reach a stopping condition.

As a simple check, let's count the words in the final story.

In [ ]:
word_count = len(final.split())
print(f"Final story word count: {word_count:,}")

## Next Steps

As an exercise, you can try to adjust the continuation prompt to take human-in-the-loop input to steer the narrative.